In [4]:
import os
import glob
import json
from tqdm import tqdm
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
from skimage.io import imread
from rasterio.warp import reproject, calculate_default_transform, Resampling
from rasterio.io import MemoryFile
from shapely.geometry import box
import numpy as np 
import geopandas as gpd

In [9]:
class GEO_PreProcessing:

    def __init__(self):
        self.global_counter = 0  # Track counter across multiple calls

    def open(self, path, no_channels=[1,2,3], transpose=True): 
        """Open an image and optionally select channels""" 
        src = rasterio.open(path) 
        img = src.read(no_channels) 

        if transpose: 
            img = img.transpose(1, 2, 0)
            return src, img
        else:
             return src,img

    
    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3]):
        
        '''
        Find the overlapping area an save both raster within intersection extent

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        
        '''
        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:

            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def align_to_ref(self, ref_src, src_src):
        """
        Align src raster to ref raster grid (CRS, resolution, extent).
        Returns aligned array and its updated transform.
        """
        # Create output array matching reference dimensions
        aligned_data = np.empty(
            (src_src.count, ref_src.height, ref_src.width), 
            dtype=src_src.dtypes[0]
        )

        # Reproject each band
        for i in range(src_src.count):
            reproject(
                source=rasterio.band(src_src, i + 1),
                destination=aligned_data[i],
                src_transform=src_src.transform,
                src_crs=src_src.crs,
                dst_transform=ref_src.transform,
                dst_crs=ref_src.crs,
                resampling=Resampling.bilinear
            )

        print("[ALIGN DONE] Aligned image to reference grid (in-memory).")
        return aligned_data, ref_src.transform

    def makeChips(self, in_pre, in_post, out_pre, out_post, no_channels=[1, 2, 3], chip_size=224, stride=224, file_prefix="", reset_counter=False):
        """
        Creates chips from pre and post images, automatically aligning or cropping if needed.
        Returns chips that are spatially aligned.
        
        Args:
            file_prefix: Optional prefix for output filenames to avoid conflicts
            reset_counter: If True, reset global counter to 0
        """
        os.makedirs(out_pre, exist_ok=True)
        os.makedirs(out_post, exist_ok=True)
        
        # Reset counter if needed
        if reset_counter:
            self.global_counter = 0
        
        counter = self.global_counter

        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:

            # Check if alignment is needed
            needs_alignment = (
                #pre_src.crs != post_src.crs or
               # pre_src.transform != post_src.transform or
                pre_src.height != post_src.height or
                pre_src.width != post_src.width
            )

            if needs_alignment:
                print("[MISMATCH] Images need alignment. Aligning post to pre...")
                
                aligned_post_data, aligned_transform = self.align_to_ref(pre_src, post_src)
                
                # Create in-memory raster for aligned post
                memfile = MemoryFile()
                aligned_post_src = memfile.open(
                    driver="GTiff",
                    height=pre_src.height,
                    width=pre_src.width,
                    count=post_src.count,
                    dtype=post_src.dtypes[0],
                    crs=pre_src.crs,
                    transform=pre_src.transform
                )
                aligned_post_src.write(aligned_post_data)
                
                print(f"[INFO] Starting chipping from aligned images ({pre_src.height}x{pre_src.width})...")
                
                for row in range(0, pre_src.height - chip_size + 1, stride):
                    for col in range(0, pre_src.width - chip_size + 1, stride):
                        window = Window(col, row, chip_size, chip_size)
                        
                        pre_chip = pre_src.read(no_channels, window=window)
                        post_chip = aligned_post_src.read(no_channels, window=window)
                        
                        chip_transform = rasterio.windows.transform(window, pre_src.transform)
                        
                        chip_meta = {
                            "driver": "GTiff",
                            "height": chip_size,
                            "width": chip_size,
                            "count": len(no_channels),
                            "dtype": pre_src.dtypes[0],
                            "crs": pre_src.crs,
                            "transform": chip_transform
                        }
                        
                        pre_filename = os.path.join(out_pre, f"{file_prefix}pre_{counter}.tif")
                        post_filename = os.path.join(out_post, f"{file_prefix}post_{counter}.tif")
                        
                        with rasterio.open(pre_filename, "w", **chip_meta) as pre_dest:
                            pre_dest.write(pre_chip)
                        
                        with rasterio.open(post_filename, "w", **chip_meta) as post_dest:
                            post_dest.write(post_chip)
                        
                        counter += 1
                
                aligned_post_src.close()
                memfile.close()
                
            else:
                # Images already aligned
                print("[INFO] Images are already aligned. Proceeding with chipping...")
                
                for row in range(0, pre_src.height - chip_size + 1, stride):
                    for col in range(0, pre_src.width - chip_size + 1, stride):
                        window = Window(col, row, chip_size, chip_size)
                        
                    
                        pre_chip = pre_src.read(no_channels, window=window)
                        post_chip = post_src.read(no_channels, window=window)
                    
                        chip_transform = rasterio.windows.transform(window, pre_src.transform)
                        
                        chip_meta = {
                            "driver": "GTiff",
                            "height": chip_size,
                            "width": chip_size,
                            "count": len(no_channels),
                            "dtype": pre_src.dtypes[0],
                            "crs": pre_src.crs,
                            "transform": chip_transform
                        }
                        
                        pre_filename = os.path.join(out_pre, f"{file_prefix}pre_{counter}.tif")
                        post_filename = os.path.join(out_post, f"{file_prefix}post_{counter}.tif")
                        
                        with rasterio.open(pre_filename, "w", **chip_meta) as pre_dest:
                            pre_dest.write(pre_chip)
                        
                        with rasterio.open(post_filename, "w", **chip_meta) as post_dest:
                            post_dest.write(post_chip)
                        
                        counter += 1

            print(f"[DONE] Created {counter - self.global_counter} chips from this image pair.")
            print(f"[TOTAL] {counter} total chips created so far.")           
            self.global_counter = counter
            
    def Mean_Std(self, FOLDER_PATH, no_channels):
        '''
        Computing the Mean and Std for the whole dataset

        Args:
            FOLDER_PATH(str): Provide the directory consisting of images
            no_channels(list): List of channel indices to read
        
        Returns:
            tuple: ((pre_mean, pre_std), (post_mean, post_std)) as numpy arrays, 
                or ((None, None), (None, None)) if no valid images
        '''

        try:
            pre_list, post_list = [], []

            pre_mean = np.zeros(len(no_channels))
            pre_std = np.zeros(len(no_channels))

            post_mean = np.zeros(len(no_channels))
            post_std = np.zeros(len(no_channels))


            for folds in os.listdir(FOLDER_PATH):
                if folds == 'SEN4FLOOD_A' or folds == 'SEN4FLOOD_B':
                    print(f'Collecting Images from-----{folds}')
                    pre_collection = os.path.join(FOLDER_PATH, folds, 'pre')
                    post_collection = os.path.join(FOLDER_PATH, folds, 'post')

                    pre_list.extend(sorted(glob.glob(os.path.join(pre_collection, '*.*'))))
                    post_list.extend(sorted(glob.glob(os.path.join(post_collection, '*.*'))))
                    
                else:
                     continue
                    
            print(f'[INFO] Found {len(pre_list)} pre-images and {len(post_list)} post-images')

            assert len(pre_list) == len(post_list), 'MISMATCH IN NUMBER OF FILES'

            skipped_count = 0
            processed_count = 0

            for preimg_name, postimg_name in tqdm(zip(pre_list, post_list),total=len(pre_list)):
                preimg = imread(preimg_name)
                postimg = imread(postimg_name)

                has_invalid_pre = np.any(~np.isfinite(preimg))
                has_invalid_post = np.any(~np.isfinite(postimg))

                if has_invalid_pre or has_invalid_post:
                    skipped_count += 1
                    print(f'\n[WARNING] Skipping image pair {skipped_count} due to invalid values:')
                    
                    if has_invalid_pre:
                        print(f'  PRE-IMAGE:  {preimg_name}')
                        print(f'    - NaN count: {np.isnan(preimg).sum()}')
                        print(f'    - Inf count: {np.isinf(preimg).sum()}')
                    
                    if has_invalid_post:
                        print(f'  POST-IMAGE: {postimg_name}')
                        print(f'    - NaN count: {np.isnan(postimg).sum()}')
                        print(f'    - Inf count: {np.isinf(postimg).sum()}')
                    
                    continue 
                
                pre_mean += np.mean(preimg, axis=(0, 1))      
                pre_std += np.std(preimg, axis=(0, 1)) 

                post_mean += np.mean(postimg, axis=(0, 1))      
                post_std += np.std(postimg, axis=(0, 1)) 

                processed_count += 1

            print(f'\n[INFO] Processed {processed_count} valid image pairs')
            print(f'[INFO] Skipped {skipped_count} image pairs with invalid values')

            # Calculate statistics
            pre_main_mean = pre_mean / processed_count
            pre_main_std = pre_std  / processed_count

            post_main_mean = post_mean / processed_count
            post_main_std = post_std / processed_count

            print(f'\n[PRE_MEAN]-----{pre_main_mean}')
            print(f'[PRE_STD]-----{pre_main_std}')

            print(f'[POST_MEAN]-----{post_main_mean}')
            print(f'[POST_STD]-----{post_main_std}')

            return None

        except Exception as e:
            print(f'[ERROR] Exception occurred: {str(e)}')
            import traceback
            traceback.print_exc()
            return None

In [10]:
process = GEO_PreProcessing()

In [11]:
# Folder path
FOLDER_PATH = r'E:\pretext_data'

process.Mean_Std(FOLDER_PATH=FOLDER_PATH,no_channels=[1,2,3])


[INFO] Found 163080 pre-images and 163080 post-images


  2%|▏         | 3259/163080 [03:12<2:34:53, 17.20it/s]


[WARNING] Skipping image pair 1 due to invalid values:
  POST-IMAGE: E:\pretext_data\SEN4FLOOD_A\post\post_3929.tif
    - NaN count: 0
    - Inf count: 1


  3%|▎         | 4850/163080 [04:49<2:36:45, 16.82it/s]


[WARNING] Skipping image pair 2 due to invalid values:
  POST-IMAGE: E:\pretext_data\SEN4FLOOD_A\post\post_54.tif
    - NaN count: 0
    - Inf count: 1


...


[WARNING] Skipping image pair 177 due to invalid values:
  PRE-IMAGE:  E:\pretext_data\SEN4FLOOD_B\pre\pre_9583.tif
    - NaN count: 0
    - Inf count: 1


 98%|█████████▊| 160439/163080 [3:58:00<04:04, 10.81it/s]


[WARNING] Skipping image pair 178 due to invalid values:
  POST-IMAGE: E:\pretext_data\SEN4FLOOD_B\post\post_9762.tif
    - NaN count: 0
    - Inf count: 1


100%|██████████| 163080/163080 [4:02:12<00:00, 11.22it/s]


[INFO] Processed 162902 valid image pairs
[INFO] Skipped 178 image pairs with invalid values

[PRE_MEAN]-----[-10.06004433 -17.2189107    0.58428808]
[PRE_STD]-----[3.2242846  3.6404165  0.80442163]
[POST_MEAN]-----[-12.03841647 -19.97186764   0.59826662]
[POST_STD]-----[4.01822834 4.7711841  0.65049576]


In [18]:
import rasterio
import numpy as np
import os

def split_sar_ratio(img,counter):

    with rasterio.open(img) as src:

        img = src.read().transpose(1, 2, 0)
        pre_meta = src.meta.copy()
        post_meta = src.meta.copy()

        #It is a ( band image were the first two band in VV and VH OF PRE EVENT and third and fourth is VV and VH OF POST EVENT)
        pre = img[:,:,:2]
        post = img[:,:,2:4]


        pre_vv = pre[:,:,0]
        pre_vh = pre[:,:,1]
        eps = 1e-6
        pre_ratio = pre_vv / (pre_vh + eps)
        pre_img = np.stack((pre_vv,pre_vh,pre_ratio),axis=-1).astype(np.float32)

        post_vv = post[:,:,0]
        post_vh = post[:,:,1]
        post_ratio = post_vv / (post_vh + eps)
        post_img = np.stack((post_vv,post_vh,post_ratio),axis=-1).astype(np.float32)

        pre_meta.update({
            'count':pre_img.shape[-1]
        })

        post_meta.update({
            'count':post_img.shape[-1]
            
        })


        pre_img = pre_img.transpose(2,0,1)
        post_img = post_img.transpose(2,0,1)


        with rasterio.open(os.path.join(r'C:\Users\b1114674\Desktop\thesis\urbanfloodsar_check\pre',f'pre_img{counter}.tif'),mode='w',**pre_meta) as pre_dest:
            pre_dest.write(pre_img)

        with rasterio.open(os.path.join(r'C:\Users\b1114674\Desktop\thesis\urbanfloodsar_check\post',f'post_img{counter}.tif'),mode='w',**post_meta) as post_dest:
            post_dest.write(post_img)

In [71]:
FOLDER_PATH = r'E:\SenForFlood_s1\DFO'

In [72]:
for folder in os.listdir(FOLDER_PATH):

    first_path = os.listdir(os.path.join(FOLDER_PATH, folder))

    for i in first_path:

        pre_path = os.path.join(FOLDER_PATH, folder,i,'s1_before_flood')
        post_path = os.path.join(FOLDER_PATH, folder,i,'s1_during_flood')

        print(pre_path)

        print(post_path)

        pre_list = sorted(glob.glob(fr'{pre_path}/*.tif'))
        post_list = sorted(glob.glob(fr'{post_path}/*.tif'))

        assert len(pre_list) == len(post_list), 'MISMATCH IN FILE NUMBER'

        for i in range(len(pre_list)):


            
            process.makeChips(
                pre_list[i],
                post_list[i],
                out_pre=r'E:\pretext_data\SEN4FLOOD_B\pre',
                out_post=r'E:\pretext_data\SEN4FLOOD_B\post',
                chip_size=224,
                stride=224
            )

E:\SenForFlood_s1\DFO\Bangladesh\DFO_4466_Bangladesh\s1_before_flood
E:\SenForFlood_s1\DFO\Bangladesh\DFO_4466_Bangladesh\s1_during_flood
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 4 total chips created so far.
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 8 total chips created so far.
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 12 total chips created so far.
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 16 total chips created so far.
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 20 total chips created so far.
[INFO] Images are already aligned. Proceeding with chipping...
[DONE] Created 4 chips from this image pair.
[TOTAL] 24 total chip

In [ ]:
import random


import numpy as np

In [84]:
# Folder path
FOLDER_PATH = r'E:\pretext_data'

pre_list, post_list = [], []

# Collect all image paths (tif, png, jpg)
for folds in os.listdir(FOLDER_PATH):
    if folds == 'SEN4FLOOD_A' or folds == 'SEN4FLOOD_B':
        pre_collection = os.path.join(FOLDER_PATH, folds, 'pre')
        post_collection = os.path.join(FOLDER_PATH, folds, 'post')

        pre_list.extend(sorted(glob.glob(os.path.join(pre_collection, '*.*'))))
        post_list.extend(sorted(glob.glob(os.path.join(post_collection, '*.*'))))

        print(f'The images from {folds} folder are collected')
        
    else:
        continue
        
# Shuffle and select subset
random.shuffle(pre_list)
random.shuffle(post_list)
new_pre_list = pre_list[:3000]
new_post_list = post_list[:3000]

pre_entropies = []
post_entropies = []

# Compute Shannon entropy for pre images
for img_path in new_pre_list:
    img = io.imread(img_path)
    
    # Handle grayscale or alpha images safely
    if img.ndim == 2:
        pre_entropies.append(shannon_entropy(img))
    else:
        # Take first 3 channels only if available (ignore alpha)
        channels = min(img.shape[2], 3)
        for i in range(channels):
            pre_entropies.append(shannon_entropy(img[:, :, i]))

# Compute Shannon entropy for post images
for img_path in new_post_list:
    img = io.imread(img_path)
    
    if img.ndim == 2:
        post_entropies.append(shannon_entropy(img))
    else:
        channels = min(img.shape[2], 3)
        for i in range(channels):
            post_entropies.append(shannon_entropy(img[:, :, i]))

print(f'[ENTROPY OF {len(new_pre_list)} PRE IMAGES]  --> {np.mean(pre_entropies):.4f}')
print(f'[ENTROPY OF {len(new_post_list)} POST IMAGES] --> {np.mean(post_entropies):.4f}')


The images from SEN4FLOOD_A folder are collected
The images from SEN4FLOOD_B folder are collected
[ENTROPY OF 3000 PRE IMAGES]  --> 10.4998
[ENTROPY OF 3000 POST IMAGES] --> 10.5234


In [53]:
FOLDER_PATH = r'E:\pretext_data'

In [140]:
pre_list, post_list = [], []
len_list = 1000
for folds in os.listdir(FOLDER_PATH):
    if folds == 'SEN4FLOOD_B' or folds == 'SEN4FLOOD_A':
        continue
    else:
        print(f'Collecting Images from-----{folds}')
        pre_collection = os.path.join(FOLDER_PATH, folds, 'pre')
        post_collection = os.path.join(FOLDER_PATH, folds, 'post')

        pre_list.extend(sorted(glob.glob(os.path.join(pre_collection, '*.*'))))
        post_list.extend(sorted(glob.glob(os.path.join(post_collection, '*.*'))))
        

In [138]:
imread(pre_list[0]).shape

(224, 224, 3)

In [141]:
for pre,post in zip(pre_list,post_list):

    if imread(pre).shape != (224,224,3) or imread(post).shape != (224,224,3):
        print(f'[FILEPATH]---{pre} and {post}')

In [125]:
pre_list, post_list = [], []
len_list = 1000
for folds in os.listdir(FOLDER_PATH):
    if folds == 'SEN4FLOOD_A' or folds == 'SEN4FLOOD_B':
        continue
    else:
        print(f'Collecting Images from-----{folds}')
        pre_collection = os.path.join(FOLDER_PATH, folds, 'pre')
        post_collection = os.path.join(FOLDER_PATH, folds, 'post')

        pre_list.extend(sorted(glob.glob(os.path.join(pre_collection, '*.*'))))
        post_list.extend(sorted(glob.glob(os.path.join(post_collection, '*.*'))))

print(len(pre_list[:len_list]))
print(len(post_list[:len_list]))

1000
1000


In [126]:
new_pre_list = pre_list[:len_list]
new_post_list = post_list[:len_list]

In [127]:
pre_mean = np.zeros(3)
pre_std = np.zeros(3)

post_mean = np.zeros(3)
post_std = np.zeros(3)

for i in range(0,len(new_pre_list)):
    pre_img = imread(new_pre_list[i])
    post_img = imread(new_post_list[i])

    pre_mean += np.mean(pre_img,axis=(0,1))
    pre_std += np.std(pre_img, axis=(0,1))

    post_mean += np.mean(post_img,axis=(0,1))
    post_std += np.std(post_img, axis=(0,1))

average_pre_mean = pre_mean/len(new_pre_list)
average_pre_std = pre_std/len(new_pre_list)


print(average_pre_mean)
print(average_pre_std)

[94.13395586 94.89801477 97.81343375]
[33.57565337 28.89160142 26.38339166]


In [132]:
for i in range(0,20):
    pre_img = imread(new_pre_list[i])
    print(f'[BEFORE NORMALIZATION]----[MIN] {pre_img.min()}, [MAX] {pre_img.max()}')
    n_pre_img =  (pre_img - average_pre_mean)/average_pre_std
    print(f'[AFTER NORMALIZATION]----[MIN] {n_pre_img.min()}, [MAX] {n_pre_img.max()}')

[BEFORE NORMALIZATION]----[MIN] 81, [MAX] 221
[AFTER NORMALIZATION]----[MIN] -0.637273401824002, [MAX] 3.7785130419089903
[BEFORE NORMALIZATION]----[MIN] 65, [MAX] 203
[AFTER NORMALIZATION]----[MIN] -1.24371552297907, [MAX] 3.242410295008208
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 208
[AFTER NORMALIZATION]----[MIN] -2.304989235000439, [MAX] 3.948945139411199
[BEFORE NORMALIZATION]----[MIN] 31, [MAX] 249
[AFTER NORMALIZATION]----[MIN] -2.494502397861398, [MAX] 5.616660972587636
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 248
[AFTER NORMALIZATION]----[MIN] -2.0396708069950966, [MAX] 5.692466237732019
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 254
[AFTER NORMALIZATION]----[MIN] -2.304989235000439, [MAX] 5.91988203316517
[BEFORE NORMALIZATION]----[MIN] 34, [MAX] 253
[AFTER NORMALIZATION]----[MIN] -2.2291839698560554, [MAX] 5.881979400592979
[BEFORE NORMALIZATION]----[MIN] 32, [MAX] 254
[AFTER NORMALIZATION]----[MIN] -2.3428918675726305, [MAX] 5.91988203316517
[BEFORE NORMALIZATION]----[MIN]

In [129]:
pre_sum = np.zeros(3)
pre_sum_sq = np.zeros(3)
pre_pixel_count = 0

post_sum = np.zeros(3)
post_sum_sq = np.zeros(3)
post_pixel_count = 0

for i in range(0,len(new_pre_list)):
    preimg = imread(new_pre_list[i])
    postimg = imread(new_post_list[i])

    pre_sum += np.sum(preimg,axis=(0,1))
    pre_sum_sq += np.sum(preimg**2, axis=(0, 1)) 
    pre_pixel_count += preimg.shape[0] * preimg.shape[1]

    post_sum += np.sum(postimg, axis=(0, 1))      
    post_sum_sq += np.sum(postimg**2, axis=(0, 1)) 
    post_pixel_count += postimg.shape[0] * postimg.shape[1]

# Calculate statistics
pre_mean = pre_sum / pre_pixel_count
pre_std = np.sqrt(pre_sum_sq / pre_pixel_count - pre_mean**2)

post_mean = post_sum / post_pixel_count
post_std = np.sqrt(post_sum_sq / post_pixel_count - post_mean**2)

In [131]:
for i in range(0,20):
    pre_img = imread(new_pre_list[i])
    print(f'[BEFORE NORMALIZATION]----[MIN] {pre_img.min()}, [MAX] {pre_img.max()}')
    n_pre_img =  (pre_img - pre_mean)/pre_std
    print(f'[AFTER NORMALIZATION]----[MIN] {n_pre_img.min()}, [MAX] {n_pre_img.max()}')

[BEFORE NORMALIZATION]----[MIN] 81, [MAX] 221
[AFTER NORMALIZATION]----[MIN] -0.5471465525944808, [MAX] 3.0965907939686015
[BEFORE NORMALIZATION]----[MIN] 65, [MAX] 203
[AFTER NORMALIZATION]----[MIN] -1.0678221605648857, [MAX] 2.6572404960441234
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 208
[AFTER NORMALIZATION]----[MIN] -1.9790044745130941, [MAX] 3.3904627326817063
[BEFORE NORMALIZATION]----[MIN] 31, [MAX] 249
[AFTER NORMALIZATION]----[MIN] -2.141715602003846, [MAX] 4.82232065460032
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 248
[AFTER NORMALIZATION]----[MIN] -1.7512088960260421, [MAX] 4.887405105596621
[BEFORE NORMALIZATION]----[MIN] 33, [MAX] 254
[AFTER NORMALIZATION]----[MIN] -1.9790044745130941, [MAX] 5.082658458585522
[BEFORE NORMALIZATION]----[MIN] 34, [MAX] 253
[AFTER NORMALIZATION]----[MIN] -1.9139200235167937, [MAX] 5.050116233087372
[BEFORE NORMALIZATION]----[MIN] 32, [MAX] 254
[AFTER NORMALIZATION]----[MIN] -2.0115467000112446, [MAX] 5.082658458585522
[BEFORE NORMALIZATION]-

FileNotFoundError: [Errno 2] No such file or directory: './data_config.yaml'

In [160]:
from configs.build_config import build_config

In [ ]:
import yaml
from yaml.loader import SafeLoader

def load_config(pretrain=True):
    with open("configs/data_config.yaml", "r") as f:
        config = yaml.load(f, Loader=yaml.FullLoader)

        if pretrain:
            return config['pretrain']
        else:
            return config


In [186]:
config = build_config(pretrain=True)

In [187]:
config['batch_size']

200

In [ ]:
#IMAGE PATCHING USING PATCHIFY FOR PNG IMAGES

from patchify import patchify
from skimage.io import imread,imsave
import os
import rasterio 

pre_path = [r'C:\Users\b1114674\Desktop\thesis\data\Building change detection dataset_add\The two-period image data\2012\whole_image\test\image\2012_test.tif',r'C:\Users\b1114674\Desktop\thesis\data\Building change detection dataset_add\The two-period image data\2012\whole_image\train\image\2012_train.tif']
post_path = [r'C:\Users\b1114674\Desktop\thesis\data\Building change detection dataset_add\The two-period image data\2016\whole_image\test\image\2016_test.tif',r'C:\Users\b1114674\Desktop\thesis\data\Building change detection dataset_add\The two-period image data\2016\whole_image\train\image\2016_train.tif']




counter = 0
for pre,post in zip(pre_path,post_path):

    pre_src = rasterio.open(pre)
    post_src = rasterio.open(post)

    pre = pre_src.read()
    post = post_src.read()

    pre_chips = patchify(pre,(224,224,3),step=224)
    post_chips = patchify(post,(224,224,3),step=224)


    for r in range(pre_chips.shape[0]):
        for c in range(pre_chips.shape[1]):

            pre_single = pre_chips[r,c,0,:,:,:]
            post_single = post_chips[r,c,0,:,:,:]

            out_pre = os.path.join(r'C:\Users\b1114674\Desktop\thesis\changedetect_check\pre', f'pre_{counter}.tif')
            out_post = os.path.join(r'C:\Users\b1114674\Desktop\thesis\changedetect_check\post', f'post_{counter}.tif')
            imsave(out_pre, pre_single)
            imsave(out_post, post_single)

            counter+=1

In [ ]:
from configs.build_config import load_config

In [ ]:
config = load_config(is_pretrain=True,type='data')

In [ ]:
config['modalities']['PRE']['SAR']['mean']